In [1]:
import heapq
import time


# ============================================================
# Utilidad: imprimir tablero
# ============================================================

def print_board(state):
    for i in range(3):
        row = state[i*3:(i+1)*3]
        print(" ".join("_" if x == 0 else str(x) for x in row))
    print()


# ============================================================
# Generación de sucesores
# ============================================================

def get_neighbors(state):
    neighbors = []

    zero_index = state.index(0)
    row, col = divmod(zero_index, 3)

    moves = [(-1,0),(1,0),(0,-1),(0,1)]  # up, down, left, right

    for dr, dc in moves:
        new_row = row + dr
        new_col = col + dc

        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_index = new_row * 3 + new_col

            new_state = list(state)
            new_state[zero_index], new_state[new_index] = new_state[new_index], new_state[zero_index]

            neighbors.append(tuple(new_state))

    return neighbors


# ============================================================
# Heurística Manhattan
# ============================================================

def manhattan(state, goal):
    distance = 0

    for value in range(1, 9):
        current_index = state.index(value)
        goal_index = goal.index(value)

        x1, y1 = divmod(current_index, 3)
        x2, y2 = divmod(goal_index, 3)

        distance += abs(x1 - x2) + abs(y1 - y2)

    return distance


# ============================================================
# Reconstrucción del camino
# ============================================================

def reconstruct_path(parent, start, goal):
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse()
    return path


# ============================================================
# A* Search
# ============================================================

def a_star(start, goal, heuristic):

    start_time = time.time()

    open_list = []
    counter = 0  # para desempatar en heap

    # g(n): costo acumulado real
    g_cost = {start: 0}

    # para reconstruir camino
    parent = {start: None}

    # insertamos estado inicial
    f_start = heuristic(start, goal)
    heapq.heappush(open_list, (f_start, counter, start))
    counter += 1

    expanded = 0

    while open_list:

        f_current, _, current = heapq.heappop(open_list)

        expanded += 1

        if current == goal:
            end_time = time.time()
            path = reconstruct_path(parent, start, goal)
            return path, {
                "expanded_nodes": expanded,
                "solution_depth": len(path) - 1,
                "time_seconds": end_time - start_time
            }

        for neighbor in get_neighbors(current):

            tentative_g = g_cost[current] + 1  # cada movimiento cuesta 1

            # Si nunca lo hemos visto o encontramos mejor camino
            if neighbor not in g_cost or tentative_g < g_cost[neighbor]:

                g_cost[neighbor] = tentative_g
                parent[neighbor] = current

                f_neighbor = tentative_g + heuristic(neighbor, goal)

                heapq.heappush(open_list, (f_neighbor, counter, neighbor))
                counter += 1

    return None, {"expanded_nodes": expanded}

In [2]:
initial = (2,8,3,
           1,6,4,
           7,0,5)

goal = (1,2,3,
        8,0,4,
        7,6,5)

print("Estado Inicial:")
print_board(initial)

print("Estado Meta:")
print_board(goal)

path, stats = a_star(initial, goal, manhattan)

print("=== RESULTADO A* ===")
print("Métricas:", stats)

if path:
    print("\nSolución paso a paso:\n")
    for step, state in enumerate(path):
        print(f"Paso {step}:")
        print_board(state)
else:
    print("No se encontró solución.")

Estado Inicial:
2 8 3
1 6 4
7 _ 5

Estado Meta:
1 2 3
8 _ 4
7 6 5

=== RESULTADO A* ===
Métricas: {'expanded_nodes': 6, 'solution_depth': 5, 'time_seconds': 0.0}

Solución paso a paso:

Paso 0:
2 8 3
1 6 4
7 _ 5

Paso 1:
2 8 3
1 _ 4
7 6 5

Paso 2:
2 _ 3
1 8 4
7 6 5

Paso 3:
_ 2 3
1 8 4
7 6 5

Paso 4:
1 2 3
_ 8 4
7 6 5

Paso 5:
1 2 3
8 _ 4
7 6 5

